**Tabla de contenido**

- [Introducción](#Introduccio)
- [Librerías](#Librerias)
- [Preprocesamiento](#Preprocesamiento)


# Introduccion

Tu tarea consiste en crear un clasificador binario que prediga si un comentario de Reddit infringe una norma específica. El conjunto de datos procede de una gran colección de comentarios moderados, con una serie de normas de subreddit, tonos y expectativas de la comunidad.

`dataset`
- **body** - el texto del comentario
- **rule** - la regla que se considera que infringe el comentario
- **subreddit** - el foro en el que se hizo el comentario
- **positive_example_{1,2}** - ejemplos de comentarios que infringen la regla
- **negative_example_{1,2}** - ejemplos de comentarios que no infringen la regla
- **rule_violation** - el objetivo binario


# Librerias

In [1]:
import os
import pandas as pd
import re

In [2]:
file_path = lambda file: os.path.join(os.getcwd(),'data/Agile Community Rules Classification',file)
train = pd.read_csv(file_path('train.csv'))
#train = train.set_index('row_id', drop=True)
#pd.set_option('display.max_colwidth', None)  # Mostrar todo el contenido de las celd
train.head(2)

,row_id,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2,rule_violation
0,0,Banks don't want you to know this! Click here ...,"No Advertising: Spam, referral links, unsolici...",Futurology,If you could tell your younger self something ...,hunt for lady for jack off in neighbourhood ht...,Watch Golden Globe Awards 2017 Live Online in ...,"DOUBLE CEE x BANDS EPPS - ""BIRDS""\n\nDOWNLOAD/...",0
1,1,SD Stream [ ENG Link 1] (http://www.sportsstre...,"No Advertising: Spam, referral links, unsolici...",soccerstreams,[I wanna kiss you all over! Stunning!](http://...,LOLGA.COM is One of the First Professional Onl...,#Rapper \n🚨Straight Outta Cross Keys SC 🚨YouTu...,[15 Amazing Hidden Features Of Google Search Y...,0


In [3]:
print(train['positive_example_1'][4])

 wow!! amazing reminds me of the old days.Well Do you desire a great spell caster and a herbal doctor to help you solve any problem you are going through? i am a proud testimony of what king favour solution temple has offered me. Contact him now at kingfavoursolutiontemple@yahoo.com You will be the next to testify.bye everyone


# Preprocesamiento

Vamos a prepar los datos para el modelo Bertweet. BERTweet es un modelo de lenguaje basado en la arquitectura BERT (Bidirectional Encoder Representations from Transformers), pero específicamente entrenado en tweets (textos de Twitter) en ingles. Está optimizado para lenguaje informal, lo que incluye jerga de redes sociales, hashtags, emoticonos, menciones (@) y ortografía no estándar (ej: "loooove"). Tiene un Tokenizador adaptado: Maneja mejor palabras repetidas ("goooool"), contracciones ("don't" → "do n't") y palabras concatenadas ("NewYork").

Este modelo está diseñado para tareas de Procesamiento de Lenguaje Natural (NLP) en redes sociales, como:

1. `Clasificación de Texto`

- Análisis de sentimiento (ej.: ¿Es un tweet positivo, negativo o neutro?).
- Detección de hate speech, spam o bullying.
- Identificación de noticias falsas (fake news) en redes sociales.

2. `Extracción de Información`

- Named Entity Recognition (NER): Identificar personas, lugares, etc., en tweets.
- Detección de temas (topic modeling) en conversaciones de Twitter.

3. `Aplicaciones Específicas`

- Moderación automática de contenido en plataformas sociales.
- Respuesta a preguntas (QA) en contextos informales.
- Generación de texto (aunque no es su enfoque principal).

Esto implica que el preprocesamiento de los textos debe realizarse de la siguiente forma:

1. Reemplazar las URL por la abreviatura `[URL]`.
2. Reemplazar las mensiones de usarios por la abreviatura `[USER]`
3. Los `Hashtag` deben dejarse tal cual como están.
4. Los emojis deben dejarse ya que ayudan al modelo a entender tono emosional o sarcasmo.
5. Se deben eliminar múltiples espacios, tabs o saltos de linea innecesarios.
6. No convertir a mayúscula o minúscula, ni eliminar los signos de puntuación. Este modelo no distigue entre mayúscula/minúscula.

In [4]:
from urlextract import URLExtract

def cleantext_toBERTweet(text):
    text = re.sub(r"\s+"," ", text).strip()                     # reemplaza múltiples espacios, tabs o saltos de línea por un solo espacio
    email_pattern = r'\b([A-Za-z0-9._%+-]+)\s*(?:@|\[at\]|\(at\)|arroba)\s*([A-Za-z0-9.-]+)\s*(?:\.|\[dot\]|\(dot\)|punto)\s*([A-Za-z]{2,})\b'
    text = re.sub(email_pattern,'[EMAIL]',text)                 # reemplaza correo electrónicos a [EMAIL]
    phone_pattern = r'(?<!\w)(?:\+?\d{1,3}|\(\+?\d{1,3}\))?(?:[-. /]?\d{2,4}){2,5}(?:[-. /]?\d{2,})\b(?:[ ]*(?:ext|xtn|x|#)[ ]*\d{1,6})?(?!\w)'
    text = re.sub(phone_pattern, '[PHONE]', text)               # Reemplaza números de teléfonos por [PHONE]

    # Reemplazo de URLs estándar detectadas por URLExtract
    extractor = URLExtract()
    urls = extractor.find_urls(text)
    for url in urls:
        text = text.replace(url, '[URL]')  
    text = re.sub(r"@\w+", " [USER] ", text)                    # reemplaza usuarios por [USER]

    # Patrón “raro” en url (/p/... .xxx)
    pattern_url_raro =  r"/[A-Za-z]/[\w-]+\.[A-Za-z]{2,4}\b"
    text = re.sub(pattern_url_raro, "[URL]", text)
    # Cualquier HTTP/HTTPS
    pattern_any_url = r"(?:https?://|://)[^\s]+"
    text = re.sub(pattern_any_url, "[URL]", text)
    # patrones raros
    pattern_scheme = r"\b[a-z][\w+.-]*://[^\s]+\b"
    text = re.sub(pattern_scheme, "[URL]", text)

    # “come” todas las aperturas de paréntesis o corchetes adyacentes antes de un token del tipo […]
    pattern =  r'([(\[])\[URL\]([)\]])'  # Captura ( [URL] ) o [ [URL] ]
    text = re.sub(pattern,'[URL]',text)
    text = re.sub(r"\*", "", text)
    text = re.sub(r'\/','',text)
    return text

Veamos ahora como quedan los texto, para esto sacaremos muestras aleatorias y las limpiaremos, esto con el fin de saber que todo está ok.

In [5]:
#train = train.set_index('row_id', drop=True)
pd.set_option('display.max_colwidth', None)  # Mostrar todo el contenido de las celd
muestra = train['body'].sample(n=10)
muestra.head(10)

1651                                                                                                      > You expunge convictions. You can't expunge arrest records.\n\nUm, no.  You can definitely get arrest records expunged or sealed.  It depends on the state as to what qualifies or how it works.
992                                                                                                                                                                               Grace hair factory -- Make beauty in your life!\n\n100% human virgin hair !!!!\nSilk and soft!!\nAccept customization!!\n
352                                        If you know what exactly you need then you don't needa prescription. You can buy it online. I buy every medication I need during last 3 years online and can recommend it. http://cheapmarketmeds.com/ is a pharmacy where you can find high-quality Wellbutrin.
1497    Is it possible for a father to give up visitation rights and any custodial claim in exchange

In [6]:
muestra = muestra.apply(cleantext_toBERTweet)
muestra.head(10)

1651                                                                                                       > You expunge convictions. You can't expunge arrest records. Um, no. You can definitely get arrest records expunged or sealed. It depends on the state as to what qualifies or how it works.
992                                                                                                                                                                                  Grace hair factory -- Make beauty in your life! 100% human virgin hair !!!! Silk and soft!! Accept customization!!
352                                                          If you know what exactly you need then you don't needa prescription. You can buy it online. I buy every medication I need during last 3 years online and can recommend it. [URL] is a pharmacy where you can find high-quality Wellbutrin.
1497    Is it possible for a father to give up visitation rights and any custodial claim in exchange for not pay

Podemos ver que la función parece funcionar correctamente. Apliquemosla al set de datos.

In [7]:
df_train_ob = train.select_dtypes(['object'])
for col in df_train_ob.columns:
    df_train_ob[col]=df_train_ob[col].apply(cleantext_toBERTweet)
df_train_ob.head()

,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2
0,Banks don't want you to know this! Click here to know more!,"No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.",Futurology,"If you could tell your younger self something different about sex, what would that be? i AM IN A CONTEST TO WIN FUNDING FOR MY SEX POSITIVE FILM: VOTE HERE: [URL]",hunt for lady for jack off in neighbourhood [URL],Watch Golden Globe Awards 2017 Live Online in HD Coverage without ADS (VIP STREAMS) = HD STREAM QUALITY >>> [WATCH LINK1][URL] = HD BROADCASTING QUALITY >>> [WATCH LINK1][URL] = Mobile Compatibility: YES = NO ADS | NO ADS | ADS =,"DOUBLE CEE x BANDS EPPS - ""BIRDS"" DOWNLOADSTREAM: [URL]"
1,SD Stream [ ENG Link 1] [URL],"No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.",soccerstreams,[I wanna kiss you all over! Stunning!][URL],"[URL] is One of the First Professional Online Gold sites. By Now, As A Game Gold Seller, we've over more than 5 yrs Of Experience And Can Pass That On To Our Customers.","#Rapper 🚨Straight Outta Cross Keys SC 🚨YouTube Search Beanie 864 Click Link BELOW To Hear Hit Single ""Ah Man"" Beanie 864 FEAT King Kota (King Kota Is Only 15!) Lit 🌡🔥👍💵💯Fr Fr [URL]","[15 Amazing Hidden Features Of Google Search You Probably Don’t Know]([URL] No one would argue the fact that Google is one of the most useful si[URL]-amazing-hidden-features-of-google-search-you-probably-dont-knowtes on the Internet. Unfortunately, most people only use about...?utm_source=reddit&utm_campaign=samreen&utm_medium=cpc)"
2,Lol. Try appealing the ban and say you won't do it again.,No legal advice: Do not offer or request legal advice.,pcmasterrace,"Don't break up with him or call the cops. If you are willing to get beat up by him to stay with him, he is obviously a real winner and you know it, so you shouldn't leave him.",It'll be dismissed: [URL] The first amendment law here is SUPER settled.,Where is there a site that still works where you can jump the GPS. Is there a FAQ to do this with iPhone or Mac?,"Because this statement of his is true. It isn't freedom of the press, it's libel. And because of this, your post serves as a promotion for Trump. Reported."
3,she will come your home open her legs with and you [URL],"No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.",sex,Selling Tyrande codes for 3€ to paypal. PM. [URL],tight pussy watch for your cock get her at this point [URL],NSFW(obviously) [URL],Good News ::Download WhatsApp 2.16.230 APK for Android – Latest Version
4,code free tyrande --->>> [Imgur][URL] for you and your friend 2 codes for 4 dollars [URL] 2$... buy one directly from here: [URL],"No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.",hearthstone,wow!! amazing reminds me of the old days.Well Do you desire a great spell caster and a herbal doctor to help you solve any problem you are going through? i am a proud testimony of what king favour solution temple has offered me. Contact him now at [EMAIL] You will be the next to testify.bye everyone,seek for lady for sex in around [URL],must be watch movie [URL],"We're streaming Pokemon Veitnamese Crystal RIGHT NOW, come watch [URL]"


Perfecto!. Con esto hemos reemplazados las URL a `[URL]`, los email a `[EMAIL]`, los numéros de telefono a [PHONE]. Ahora lo que sigue es eliminar el ruido que no es útil para el modelo. Por ejemplo:

- Cadenas sin letras (solo números + símbolos, ej: 43567&%^*).
- Símbolos repetidos o combinaciones sin sentido (ej: !!??, %%%, --==).
- Caracteres especiales sueltos (ej: &, ^, * si no están pegados a palabras).

In [36]:
def eliminar_ruido(text):
    pattern = r'FindSexToday\s*\.\s*com'
    text = re.sub(pattern, '[URL]', text, flags=re.IGNORECASE)
    text = re.sub(r'\.{1,}', '.', text)
    text = re.sub(r'\!{1,}','!',text) # Cualquier secuencia de 4 o más ! se reemplazará por !!! (tres signos de exclamación).
    text = re.sub(r'\={2,}', '=', text)
    text = re.sub(r'\-{2,}','-',text)
    text = re.sub(r'\<{2,}','<',text)
    text = re.sub(r'\>{2,}','>',text)
    text = re.sub(r'\. \.\.\.', '.', text)  # Reemplaza ". ..." por "."
    text =re.sub(r'\.{2}', '. ', text)  # Solo cambia ".." → "."
    text = re.sub(r'\?{2,}','?',text)
    text = re.sub(r'(?<=[a-zA-Z])[.,](?=[a-zA-Z])', r'\g<0> ', text)  # gregar un espacio después de los signos de puntuación básicos 
    text = re.sub(r'\s+([.,;])', r'\1', text)  # Elimina espacios antes los signo ,.;
    text = re.sub(r'\|', '', text)
    text = re.sub(r'\s+', ' ', text)

    text = re.sub(r'\b\d{1,3}-\d{2,3}-\d{2,4}(?:-\d{2,4})?\b', '[PHONE]', text)
    text = text.strip()
    return text
    

In [48]:
muestra = df_train_ob['body'].sample(n=10)
muestra.head(10)

1415                                                                                                                                                                                                                                                                                                                                                                                           We're streaming Pokemon Veitnamese Crystal RIGHT NOW, come watch [URL]
85                                                                                                                                                                                                                                                                                                                                                                                                   At parties, I only put rohypnol in my own drinks, for attention.
1939                                                                                        

In [45]:
muestra = muestra.apply(eliminar_ruido)
muestra.head(10)

558                              It was a mistake involving the police because they don't give a shit about this issue. The only way you have a chance to get 'justice' is to do exactly what your girl did and snoop around. Spy on his bitch ass. Knock if door in, bust his mouth and get your stuff. He most likely won't call the cops because he's a criminal. But. it's a little hard to do it that way because you filed a report and if he does get fucked up, they'll probably tie it to you.
33                                                                                                                                                                                                                                                                                                                                                                                                                                                  this girl gonna be in ur house and lay on you [URL]
1660                    

In [ ]:
texto = "==\nWatch UFC LIVE STREAM FREE : [UFCHDTV.COM](https://UFCHDTV.COM)\n==\n\n;'\n''''\n\n\n-0\n43567&%^*"
"FindSexToday . com - free girls for sex worldwide 16rTAFMbhxgx30!"

In [ ]:
texto ="FindSexToday . com - free girls for sex worldwide 16rTAFMbhxgx30! [5 most searched female athletes][URL] 43567&%^* camino,camino. Camino"

In [ ]:
texto = str(texto)
texto = eliminar_ruido(texto)
print(texto)

In [ ]:
texto = "SD // [URL] // Lang: nl"
patron = r'\/'
texto_limpio = re.sub(patron, '', texto)

print(texto_limpio)